<a href="https://colab.research.google.com/github/AarohiAnalyzes/rag-30-days/blob/main/Day_02_Basic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage:1 Basic LLM generation**

In [20]:
!pip install genai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 2.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tiktoken (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tiktoken
Failed to build tiktoken
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tiktoken)


In [21]:
from google import genai

In [10]:
# Access the secret API key

from google.colab import userdata
API_Key = userdata.get('Gemini_API_Key')
print("API key is loaded:", API_Key is not None)

API key is loaded: True


In [12]:
# create the Gemini client which will communicate with the Gemini Api
client = genai.Client(api_key= API_Key)

In [13]:
print(client)

In [16]:
# List available models
# show only models that can generate text
for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [17]:
# make the first gemini request

response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = "Explain what is docker in one sentence."
)

In [19]:
print(response.text)

Docker is a software platform that packages applications and all their dependencies into lightweight, portable containers so they can run consistently across any computing environment.


# **Stage:2 Manual Context**

In [22]:
context = """
Docker is a platform that packages applications and their dependencies
into lightweight, portable containers.
"""

question = "What is Docker?"

In [23]:
# create a prompt using context and question
prompt = f"""
Context:
{context}

Question:
{question}
"""

In [24]:
print(prompt)


Context:

Docker is a platform that packages applications and their dependencies
into lightweight, portable containers.


Question:
What is Docker?



In [25]:
# Now call gemini

response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt
)

print(response.text)

Based on the provided context, Docker is a platform that packages applications and their dependencies into lightweight, portable containers.


# **Stage:3 Retrievel + Context + LLM**

In [26]:
# create documents list

documents = [
    "Mars is the fourth planet from the Sun and has a thin atmosphere.",
    "Jupiter is the largest planet in our solar system and is known for its Great Red Spot.",
    "The Moon is Earth's natural satellite and completes an orbit around Earth in about 27 days.",
    "Saturn is famous for its extensive system of icy rings.",
    "Venus is the hottest planet in our solar system because of its thick atmosphere.",
    "The Sun is a star at the center of our solar system and provides Earth with energy.",
    "Mercury is the smallest planet and the closest planet to the Sun.",
    "Neptune is a distant ice giant known for its strong winds."
]

In [27]:
from sentence_transformers import SentenceTransformer

In [28]:
# load the model that converts document text into vectors
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
# Create document embeddings
document_embeddings = model.encode(documents)
print(document_embeddings.shape)

(8, 384)


In [32]:
# Create a query
query = "Which planet is the largest in our solar system?"

# embed the query
query_embedding = model.encode(query)
print(query_embedding.shape)

(384,)


In [34]:
# calculate the cosine similarity to find the most top-3 similar documents

from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    [query_embedding],
    document_embeddings
)

print(similarity_scores)


[[0.4782216  0.6611974  0.23022373 0.3925048  0.5396116  0.4438303
  0.547436   0.38399798]]


In [36]:
# Rank the documents

import numpy as np
ranked_indices = np.argsort(similarity_scores[0])[::-1]
print(ranked_indices)

[1 6 4 0 5 3 7 2]


In [38]:
# print the top-3 semantic similar documents
for i in ranked_indices[::3]:
  print(documents[i])

Jupiter is the largest planet in our solar system and is known for its Great Red Spot.
Mars is the fourth planet from the Sun and has a thin atmosphere.
Neptune is a distant ice giant known for its strong winds.


# **Stage:3 Create the context**

In [42]:
top_documents = []

for i in ranked_indices[:3]:
  document = documents[i]
  top_documents.append(document)

print(top_documents)

['Jupiter is the largest planet in our solar system and is known for its Great Red Spot.', 'Mercury is the smallest planet and the closest planet to the Sun.', 'Venus is the hottest planet in our solar system because of its thick atmosphere.']


In [43]:
context = "\n".join(top_documents)

print(context)

Jupiter is the largest planet in our solar system and is known for its Great Red Spot.
Mercury is the smallest planet and the closest planet to the Sun.
Venus is the hottest planet in our solar system because of its thick atmosphere.


In [44]:
# Build the RAG Prompt

prompt = f"""
Context:
{context}

Question:
{query}
"""

print(prompt)


Context:
Jupiter is the largest planet in our solar system and is known for its Great Red Spot.
Mercury is the smallest planet and the closest planet to the Sun.
Venus is the hottest planet in our solar system because of its thick atmosphere.

Question:
Which planet is the largest in our solar system?



# **Stage:4 Generate the response from Gemini**

In [45]:
response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt
)

print(response.text)

Based on the context provided, **Jupiter** is the largest planet in our solar system.


In [46]:
# response without context
response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = query
)

print(response.text)

The largest planet in our solar system is **Jupiter**. 

It is a gas giant with a mass more than two and a half times that of all the other planets in the solar system combined, and it is large enough that over 1,300 Earths could fit inside it.
